In [ ]:
%%capture
!pip install ucimlrepo catboost xgboost

In [ ]:
from ucimlrepo import fetch_ucirepo
bank_marketing = fetch_ucirepo(id=222)

In [ ]:
cat_cols_names = list(bank_marketing.variables.query("type in ['Categorical', 'Date', 'Binary']")["name"].values)
cat_cols_names.remove('y')
print("Cat_cols_names:", cat_cols_names)

Cat_cols_names: ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'day_of_week', 'month', 'poutcome']


In [ ]:
bank_marketing.variables

,name,role,type,demographic,description,units,missing_values
0,age,Feature,Integer,Age,None,None,no
1,job,Feature,Categorical,Occupation,"type of job (categorical: 'admin.','blue-colla...",None,no
2,marital,Feature,Categorical,Marital Status,"marital status (categorical: 'divorced','marri...",None,no
3,education,Feature,Categorical,Education Level,"(categorical: 'basic.4y','basic.6y','basic.9y'...",None,no
4,default,Feature,Binary,None,has credit in default?,None,no
5,balance,Feature,Integer,None,average yearly balance,euros,no
6,housing,Feature,Binary,None,has housing loan?,None,no
7,loan,Feature,Binary,None,has personal loan?,None,no
8,contact,Feature,Categorical,None,contact communication type (categorical: 'cell...,None,yes
9,day_of_week,Feature,Date,None,last contact day of the week,None,no


In [ ]:
X = bank_marketing.data.features
y = bank_marketing.data.targets['y'].map({"no": 0, "yes": 1})

In [ ]:
(X.isna().mean() * 100).round(2).sort_values(ascending=False)

,0
poutcome,81.75
contact,28.80
education,4.11
job,0.64
default,0.00
balance,0.00
age,0.00
marital,0.00
loan,0.00
housing,0.00


In [ ]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.metrics import classification_report
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

In [ ]:
RANDOM_STATE = 42
catboost_model = CatBoostClassifier(
    loss_function="Logloss",
    random_state=RANDOM_STATE,
    verbose=0
)
xgboost_model = XGBClassifier(
    objective="binary:logistic",
    eval_metric = "logloss",
    random_state=RANDOM_STATE,
    missing = float("nan"),
    n_jobs = -1
)
preprocess = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown="ignore"), cat_cols_names)
])
xgboost_pipeline = Pipeline([
    ("preprocess", preprocess),
    ("xgbclassifier", xgboost_model)
])

In [ ]:
X[cat_cols_names] = X[cat_cols_names].astype(str).copy()
X[["default", "housing", "loan"]] = X[["default", "housing", "loan"]].astype(str).copy()
X.dtypes

/tmp/ipykernel_27458/3519782729.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X[cat_cols_names] = X[cat_cols_names].astype(str).copy()
/tmp/ipykernel_27458/3519782729.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X[["default", "housing", "loan"]] = X[["default", "housing", "loan"]].astype(str).copy()


,0
age,int64
job,object
marital,object
education,object
default,object
balance,int64
housing,object
loan,object
contact,object
day_of_week,object


In [ ]:
X.sample(4)

,age,job,marital,education,default,balance,housing,loan,contact,day_of_week,month,duration,campaign,pdays,previous,poutcome
17229,38,management,single,tertiary,no,8,yes,no,cellular,28,jul,219,2,-1,0,nan
34163,36,admin.,single,secondary,no,728,no,no,cellular,30,apr,230,1,-1,0,nan
2691,49,housemaid,divorced,primary,no,198,yes,no,nan,13,may,205,3,-1,0,nan
32584,24,blue-collar,married,secondary,no,9,yes,yes,cellular,17,apr,468,1,-1,0,nan


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [ ]:
cat_params = {
    "iterations": [500, 1000, 1500],
    "learning_rate": [0.03, 0.05, 0.1],
    "depth": [4, 6, 8],
    "l2_leaf_reg": [1, 3, 5, 9]
}
xgboost_params = {
    "xgbclassifier__n_estimators": [300, 500, 800],
    "xgbclassifier__learning_rate": [0.03, 0.05, 0.1],
    "xgbclassifier__max_depth": [3, 5, 7],
    "xgbclassifier__subsample": [0.7, 0.8, 1.0],
    "xgbclassifier__colsample_bytree": [0.7, 0.8, 1.0],
    "xgbclassifier__reg_lambda": [0.1, 1, 10]
}

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

In [ ]:
grid_cat = GridSearchCV(
    estimator = catboost_model,
    param_grid = cat_params,
    cv = cv,
    scoring = "f1",
    n_jobs = -1,
    refit=False,
    verbose=2
)

In [ ]:
grid_xgb = GridSearchCV(
    estimator = xgboost_pipeline,
    param_grid = xgboost_params,
    cv = cv,
    scoring = "f1",
    n_jobs = -1,
    refit=False,
    verbose=2
)

In [ ]:
grid_cat.fit(X_train[:500], y_train[:500], cat_features = cat_cols_names)
grid_cat.best_params_

Fitting 5 folds for each of 108 candidates, totalling 540 fits


{'depth': 4, 'iterations': 1500, 'l2_leaf_reg': 9, 'learning_rate': 0.1}

In [ ]:
grid_xgb.fit(X_train[:500], y_train[:500])
grid_xgb.best_params_

In [ ]:
best_catboost_model = CatBoostClassifier(
    loss_function="Logloss",
    random_state=RANDOM_STATE,
    verbose=0,
    cat_features=cat_cols_names
)
best_xgboost_model = XGBClassifier(
    objective="binary:logistic",
    eval_metric = "logloss",
    random_state=RANDOM_STATE,
    missing = float("nan"),
    n_jobs = -1
)
preprocess = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown="ignore"), cat_cols_names)
])
best_xgboost_pipeline = Pipeline([
    ("preprocess", preprocess),
    ("xgbclassifier", best_xgboost_model)
])

In [ ]:
best_catboost_model.fit(X_train, y_train)

In [ ]:
best_xgboost_pipeline.fit(X_train, y_train)

In [ ]:
pred_cat = best_catboost_model.predict(X_test)

In [ ]:
pred_xgb = best_xgboost_pipeline.predict(X_train)

In [ ]:
classification_report(y_test, pred_cat)

In [ ]:
classification_report(y_test, pred_xgb)